# Benchmark Temporal Splitting

This notebook implements the benchmark using **Temporal (Chronological)** splitting:
- **Train**: 0% - 70% of timeline
- **Val**: 70% - 85% of timeline  
- **Test**: 85% - 100% of timeline

This prevents data leakage and tests true extrapolation capability.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

# Import from src
from src.data_processing import generate_cyclical_features, scale_data_selectively
from src.splitting import get_temporal_splits
from src.models import TinyTGT, GlobalFitLocalApplySARIMA
from src.evaluation import RollingDataset, prepare_xgb_data, run_evaluation, print_metrics, get_scaling_factors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
os.getcwd()

'd:\\Data\\studium\\Master\\MA_Code\\Thesis_Repo\\phase1_baseline'

In [3]:
CONFIG = {
    # Date_Range that will be used to generate the temporal features 
    "START_DATE": "2019-01-01",
    "FREQUENCY": "H",                 
    
    # Dataset
    "DATA_PATH": "../../data/data_out/Three_Years_2019-2021/case118_ieee/raw", #!
    
    "USE_SUBSET": False,  #! Set to False for full dataset
    "SUBSET_PERCENT": 0.01,  # Use 1% of data (fast testing)

    # Features
    "USE_ONLY_LOAD": True,              # Only use active load P
    "BINARY_ADJACENCY": True,           # True = Graph is 0/1 (no admittance)

    #TGT
    "BATCH_SIZE": 32,                  # Batch Size
    "EPOCHS": 10,                      # Number of Epochs
    
    # Forecasting  (Triebe et al. Logic)
    "INPUT_WINDOW": 168,                # Lookback: 7 Tage
    "FORECAST_HORIZON": 1,             # If 33: Predict: Rest of Day (9h) + Next Day (24h)
    "EVAL_HOUR": None,                    # (None | Int) If 14: forecasting is done always at 2pm    
    # Baselines
    "SNAIVE_LAG": 48                    # Benchmark: value 48h ago
}

In [4]:
# Load Bus and Branch DataFrames
# Adjusted paths relative to the notebook location
bus_df = pd.read_parquet(CONFIG["DATA_PATH"]+'/bus_data.parquet')
branch_df = pd.read_parquet(CONFIG["DATA_PATH"]+'/branch_data.parquet')

# Build Edge Index from Branch Data
structure = branch_df[branch_df['load_scenario_idx'] == 0].sort_values("idx")
edge_index = list(structure[['from_bus', 'to_bus']].itertuples(index=False, name=None))

# print dims
m = len(edge_index)
n=bus_df['bus'].nunique()
print(f"Topology Loaded: n={n} Nodes, m={m} Edges")

# X: Input Loads (Active Power 'Pd')
loads_pivot = bus_df.pivot(index='load_scenario_idx', columns='bus', values='Pd').fillna(0)
# Y: Target Flows (Active Power Flow 'pf')
flows_pivot = branch_df.pivot(index='load_scenario_idx', columns='idx', values='pf').fillna(0)

# Convert to Numpy Arrays
loads_matrix = loads_pivot.values.astype(np.float32)
flows_matrix = flows_pivot.values.astype(np.float32)
print(f"shapes: Loads: {loads_matrix.shape}, Flows: {flows_matrix.shape}")

# Apply subset for testing
if CONFIG["USE_SUBSET"]:
    subset_size = int(len(loads_matrix) * CONFIG["SUBSET_PERCENT"])
    loads_matrix = loads_matrix[:subset_size]
    flows_matrix = flows_matrix[:subset_size]
    print(f"Using {subset_size} samples ({CONFIG['SUBSET_PERCENT']*100:.1f}%)")

# Adjacency mask (neighbors + self)
A = np.zeros((n,n), dtype=bool)
for (i,j) in edge_index:
    A[i,j]=True; A[j,i]=True
for i in range(n): A[i,i]=True
A_mask = torch.from_numpy(A).to(device)

Topology Loaded: n=118 Nodes, m=186 Edges
shapes: Loads: (26280, 118), Flows: (26280, 186)


In [5]:
# --- Prepare Input Tensor X --- 
# 1. gen Time Features 
T = loads_matrix.shape[0]
time_features_global = generate_cyclical_features(T, CONFIG["START_DATE"], CONFIG["FREQUENCY"]) # [T, 6]
print(f"Time Features generated. Shape: {time_features_global.shape}")

# 2. gen input tensor X
load_tensor = loads_matrix[:, :, np.newaxis]  # [T, N] -> [T, N, 1]
time_tensor_expanded = np.broadcast_to(
    time_features_global[:, np.newaxis, :], 
    (time_features_global.shape[0], loads_matrix.shape[1], time_features_global.shape[1])
) # [T, 6] -> [T, N, 6]  (*read-only view)
X = np.concatenate([load_tensor, time_tensor_expanded], axis=2) # [T, N, 1] + [T, N, 6] -> [T, N, 7]
print(f"Final Input Tensor Shape: {X.shape} (Time, Buses, Features)")

# --- Split Data ---
train_idx, val_idx, test_idx = get_temporal_splits(T)
print(f"Train Hours: {len(train_idx)}")
print(f"Val Hours:   {len(val_idx)}")
print(f"Test Hours:  {len(test_idx)}")

# --- 3. Scaling (Only P not the sin/cos transformed time features) ---
scaled_tensor, scaler = scale_data_selectively(X, train_idx)
print(f"Scaled Mean (Load): {scaled_tensor[:,:,0].mean():.4f}")
print(f"Scaled Mean (Hour_Sin): {scaled_tensor[:,:,1].mean():.4f} (Should be near 0 but unscaled)")

d:\Data\studium\Master\MA_Code\Thesis_Repo\phase1_baseline\src\data_processing.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start=start_date, periods=T, freq=frequency)


Time Features generated. Shape: (26280, 6)
Final Input Tensor Shape: (26280, 118, 7) (Time, Buses, Features)
Train Hours: 18396
Val Hours:   3942
Test Hours:  3942
Scaled Mean (Load): 0.0023
Scaled Mean (Hour_Sin): 0.0000 (Should be near 0 but unscaled)


In [6]:
# --- Create Datasets & Loaders ---
train_dataset = RollingDataset(scaled_tensor, train_idx, CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])
val_dataset   = RollingDataset(scaled_tensor, val_idx,   CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

# Test One Batch
x_batch, y_batch = next(iter(train_loader))
print(f"Batch Shapes:")
print(f"X (Input):  {x_batch.shape}  -> [Batch, {CONFIG['INPUT_WINDOW']}, 14, 7]")
print(f"Y (Target): {y_batch.shape}  -> [Batch, {CONFIG['FORECAST_HORIZON']}, 14, 7]")

Batch Shapes:
X (Input):  torch.Size([32, 168, 118, 7])  -> [Batch, 168, 14, 7]
Y (Target): torch.Size([32, 1, 118, 7])  -> [Batch, 1, 14, 7]


In [7]:
# --- XGBoost Training ---
X_train_xgb, Y_train_xgb = prepare_xgb_data(
    scaled_tensor, 
    train_idx, 
    CONFIG["INPUT_WINDOW"], 
    CONFIG["FORECAST_HORIZON"],
    step_size=1
)
print(f"XGB Train Shape: {X_train_xgb.shape}")

xgb_estimator = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    objective='reg:squarederror',
    device='cuda',       
    tree_method='hist',
    n_jobs=4
)

xgb_model = MultiOutputRegressor(xgb_estimator, n_jobs=1)

print("Training Global XGBoost...")
xgb_model.fit(X_train_xgb, Y_train_xgb)
print("XGBoost Training Complete.")

Allocating RAM for 2150904 samples...


Building XGB Dataset: 100%|██████████| 18228/18228 [00:05<00:00, 3406.40it/s]


XGB Train Shape: (2150904, 175)
Training Global XGBoost...
XGBoost Training Complete.


In [ ]:
# --- TinyTGT Training ---
torch.cuda.empty_cache() 

tgt_model = TinyTGT(n_nodes=14, d_model=64, n_heads=4, in_feat=7, out_steps=CONFIG["FORECAST_HORIZON"]).to(device)
opt = optim.Adam(tgt_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for ep in range(1, CONFIG["EPOCHS"]+1):
    tgt_model.train()
    train_loss, batch_count = 0.0, 0
    
    for xb, yb in train_loader:
        xb = xb.float().to(device) 
        yb = yb.float().to(device) 
        
        opt.zero_grad()
        yhat = tgt_model(xb, A_mask) 
        loss = loss_fn(yhat, yb[..., 0])
        loss.backward()
        opt.step()
        
        train_loss += loss.item()
        batch_count += 1
    
    # Validation
    tgt_model.eval()
    val_loss, val_batches = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.float().to(device)
            y_true = yb[..., 0].float().to(device)
            yhat = tgt_model(xb, A_mask)
            val_loss += loss_fn(yhat, y_true).item()
            val_batches += 1
            
    print(f"Epoch {ep}: Train MSE={train_loss/batch_count:.6f} | Val MSE={val_loss/val_batches:.6f}")

In [ ]:
# --- SARIMA Global Fit ---
# For temporal split: use ALL training data (contiguous, no gaps)
sarima_model = GlobalFitLocalApplySARIMA(order=(2, 0, 0), seasonal_order=(1, 0, 1, 24))
sarima_model.fit(scaled_tensor.astype(np.float32), train_idx, None)

Fitting Bus Models:   0%|          | 0/14 [00:00<?, ?it/s]d:\Data\studium\Master\MA_Code\thesis_env\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
Fitting Bus Models: 100%|██████████| 14/14 [10:08<00:00, 43.47s/it]


In [ ]:
# --- Evaluation ---
denom_mae, denom_mse = get_scaling_factors(scaled_tensor, train_idx, scaler, m=24)
print(f"Scaling Baseline (Train): MAE={denom_mae:.4f}, MSE={denom_mse:.4f}")

results, starts = run_evaluation(scaled_tensor, test_idx, xgb_model, tgt_model, sarima_model, A_mask, scaler, CONFIG, device=device)

Scaling Baseline (Train): MAE=0.6974, MSE=2.8498
Evaluating 3942 days...


  0%|          | 0/3942 [00:00<?, ?it/s]d:\Data\studium\Master\MA_Code\thesis_env\Lib\site-packages\xgboost\core.py:751: UserWarning: [12:45:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
100%|██████████| 3942/3942 [51:14<00:00,  1.28it/s]    


In [ ]:
print_metrics(results, denom_mae, denom_mse)


=== FINAL BENCHMARK RESULTS ===
Model      | RMSE [MW]  | MAE [MW]   | MAPE [%]   | MASE       | MSSE      
---------------------------------------------------------------------------
SNAIVE     | 2.2788     | 0.9459     | 10.15%     | 1.3564     | 1.8222
XGB        | 0.5836     | 0.2241     | 2.60%     | 0.3214     | 0.1195
SARIMA     | 0.2195     | 0.0867     | 0.95%     | 0.1243     | 0.0169
TinyTGT    | 0.5605     | 0.3011     | 4.28%     | 0.4318     | 0.1102
---------------------------------------------------------------------------


In [ ]:


# Flatten and reshape for Parquet export
n_evals, n_buses, n_horizon = results["true"].shape

# Save the original scenario indices for the CSV/Parquet index
scenario_indices = loads_pivot.index.values

# Create a dataframe with one row per (evaluation_step, bus, horizon_step)
data_list = []

for eval_idx, t in enumerate(starts):  # t is the time index in the array
    scenario_id = scenario_indices[t]  # Convert to original load_scenario_idx
    for bus_id in range(n_buses):
        for h in range(n_horizon):
            row = {
                'load_scenario_idx': scenario_id,
                'bus_id': bus_id,
                'horizon_step': h,
                'true': results["true"][eval_idx, bus_id, h],
                'xgb': results["xgb"][eval_idx, bus_id, h],
                'snaive': results["snaive"][eval_idx, bus_id, h],
                'tgt': results["tgt"][eval_idx, bus_id, h],
                'sarima': results["sarima"][eval_idx, bus_id, h],
            }
            data_list.append(row)

df_results = pd.DataFrame(data_list)
df_results.to_parquet('benchmark_results.parquet', index=False, compression='snappy')
print(f"Results saved to benchmark_results.parquet")
print(f"Shape: {df_results.shape}")
print(df_results.head())

Results saved to benchmark_results.parquet
Shape: (55188, 8)
   load_scenario_idx  bus_id  horizon_step          true        xgb  \
0            22338.0       0             0 -3.741942e-07   0.045107   
1            22338.0       1             0  1.847332e+01  18.486085   
2            22338.0       2             0  6.205582e+01  67.192438   
3            22338.0       3             0  3.655808e+01  36.001772   
4            22338.0       4             0  4.591058e+00   4.388583   

         snaive        tgt        sarima  
0 -3.741942e-07  -0.069498 -3.741941e-07  
1  1.784562e+01  19.021614  1.863006e+01  
2  6.799315e+01  65.594363  6.257194e+01  
3  3.353556e+01  37.181816  3.638837e+01  
4  3.554487e+00   3.477875  4.577746e+00  


In [ ]:
# --- Output Format Preview ---
print("\n=== OUTPUT FORMAT PREVIEW ===")
print(f"Expected shape: ({len(test_idx) * 14 * CONFIG['FORECAST_HORIZON']}, 8)")
print("\nColumns: 'load_scenario_idx', 'bus_id', 'horizon_step', 'true', 'xgb', 'snaive', 'tgt', 'sarima'")
print("\nExample rows (what you'll get):")
print("""
   load_scenario_idx  bus_id  horizon_step      true       xgb     snaive       tgt    sarima
0              1000        0              0  1245.34   1250.12   1248.90  1246.78   1244.56
1              1000        1              0   856.43    860.15    858.23   857.92    855.34
2              1000        2              0   923.12    920.45    925.67   922.34    924.11
...
N           1001         13              0  1567.89   1570.23   1565.43  1568.92   1566.15
""")
print("\nData types:")
print("  - load_scenario_idx: int64 (matches input parquet)")
print("  - bus_id: int64 (0-13)")
print("  - horizon_step: int64 (0 to FORECAST_HORIZON-1)")
print("  - true/xgb/snaive/tgt/sarima: float64 (MW, denormalized)")
print(f"\nWith 1% subset + FORECAST_HORIZON=1:")
print(f"  Estimated rows: ~{(len(test_idx) // 100) * 14} rows")
print(f"  File size: ~{(len(test_idx) // 100) * 14 * 8 / 1024:.1f} KB (parquet compressed)")



=== OUTPUT FORMAT PREVIEW ===
Expected shape: (55188, 8)

Columns: 'load_scenario_idx', 'bus_id', 'horizon_step', 'true', 'xgb', 'snaive', 'tgt', 'sarima'

Example rows (what you'll get):

   load_scenario_idx  bus_id  horizon_step      true       xgb     snaive       tgt    sarima
0              1000        0              0  1245.34   1250.12   1248.90  1246.78   1244.56
1              1000        1              0   856.43    860.15    858.23   857.92    855.34
2              1000        2              0   923.12    920.45    925.67   922.34    924.11
...
N           1001         13              0  1567.89   1570.23   1565.43  1568.92   1566.15


Data types:
  - load_scenario_idx: int64 (matches input parquet)
  - bus_id: int64 (0-13)
  - horizon_step: int64 (0 to FORECAST_HORIZON-1)
  - true/xgb/snaive/tgt/sarima: float64 (MW, denormalized)

With 1% subset + FORECAST_HORIZON=1:
  Estimated rows: ~546 rows
  File size: ~4.3 KB (parquet compressed)


In [ ]:
# --- How to Use the Output File ---
# Load the results back:
# df = pd.read_parquet('benchmark_results.parquet')

# Filter by bus:
# bus_0 = df[df['bus_id'] == 0]

# Filter by scenario:
# scenario_results = df[df['load_scenario_idx'] == 1000]

# Get all predictions for a specific bus at a specific scenario:
# bus_0_scenario_1000 = df[(df['bus_id'] == 0) & (df['load_scenario_idx'] == 1000)]

# Compare models (MAE per bus):
# mse_by_model = df.groupby('bus_id').apply(lambda x: {
#     'xgb_mae': (x['true'] - x['xgb']).abs().mean(),
#     'tgt_mae': (x['true'] - x['tgt']).abs().mean(),
#     'sarima_mae': (x['true'] - x['sarima']).abs().mean(),
# })

print("Output file ready to use for analysis!")


Output file ready to use for analysis!
